In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    StratifiedKFold, GridSearchCV, RandomizedSearchCV,
    cross_val_score, validation_curve, learning_curve
)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ===============================
# 1. LOAD VÀ CHUẨN BỊ DATA
# ===============================
def load_and_prepare_data():
    """Load và chuẩn bị dữ liệu"""
    # Load dataset
    data = load_breast_cancer()
    X, y = data.data, data.target

    # Chia train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test


In [ ]:
# ===============================
# 2. OPTIMIZATION VỚI STRATIFIEDKFOLD
# ===============================
def optimize_with_stratified_kfold(X_train, y_train):
    """Optimize AdaBoost với StratifiedKFold"""
    print("=" * 60)
    print("OPTIMIZATION VỚI STRATIFIEDKFOLD")
    print("=" * 60)

    # Định nghĩa parameter grid
    param_grid = {
        'n_estimators': [50, 100, 200, 300],
        'learning_rate': [0.01, 0.1, 0.5, 1.0, 1.5],
        'estimator__max_depth': [1, 2, 3, 4],
        'algorithm': ['SAMME', 'SAMME.R']
    }

    # Tạo base estimator
    base_estimator = DecisionTreeClassifier(random_state=42)

    # Tạo AdaBoost classifier
    ada = AdaBoostClassifier(
        estimator=base_estimator,
        random_state=42
    )

    # Tạo StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Grid Search với StratifiedKFold
    grid_search = GridSearchCV(
        estimator=ada,
        param_grid=param_grid,
        cv=skf,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    # Fit
    grid_search.fit(X_train, y_train)

    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_:.4f}")

    return grid_search.best_estimator_

In [ ]:
# ===============================
# 3. RANDOMIZED SEARCH OPTIMIZATION
# ===============================
def optimize_with_randomized_search(X_train, y_train):
    """Optimize AdaBoost với RandomizedSearchCV"""
    print("\n" + "=" * 60)
    print("OPTIMIZATION VỚI RANDOMIZED SEARCH")
    print("=" * 60)

    # Định nghĩa parameter distributions
    param_distributions = {
        'n_estimators': randint(50, 500),
        'learning_rate': uniform(0.01, 1.99),  # từ 0.01 đến 2.0
        'estimator__max_depth': randint(1, 6),
        'estimator__min_samples_split': randint(2, 20),
        'estimator__min_samples_leaf': randint(1, 10)
    }

    # Tạo base estimator
    base_estimator = DecisionTreeClassifier(random_state=42)

    # Tạo AdaBoost classifier
    ada = AdaBoostClassifier(
        estimator=base_estimator,
        random_state=42
    )

    # StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Randomized Search
    random_search = RandomizedSearchCV(
        estimator=ada,
        param_distributions=param_distributions,
        n_iter=100,  # số lần thử random
        cv=skf,
        scoring='accuracy',
        n_jobs=-1,
        random_state=42,
        verbose=1
    )

    # Fit
    random_search.fit(X_train, y_train)

    print(f"Best parameters: {random_search.best_params_}")
    print(f"Best CV score: {random_search.best_score_:.4f}")

    return random_search.best_estimator_

In [ ]:
# ===============================
# 4. MANUAL HYPERPARAMETER TUNING
# ===============================
def optimize_manually(X_train, y_train):
    """Manual optimization với validation curves"""
    print("\n" + "=" * 60)
    print("MANUAL OPTIMIZATION VỚI VALIDATION CURVES")
    print("=" * 60)

    # StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # 1. Tối ưu n_estimators
    n_estimators_range = [10, 50, 100, 200, 300, 500]
    train_scores, val_scores = validation_curve(
        AdaBoostClassifier(random_state=42),
        X_train, y_train,
        param_name='n_estimators',
        param_range=n_estimators_range,
        cv=skf,
        scoring='accuracy'
    )

    best_n_estimators = n_estimators_range[np.argmax(val_scores.mean(axis=1))]
    print(f"Best n_estimators: {best_n_estimators}")

    # 2. Tối ưu learning_rate
    learning_rate_range = [0.01, 0.1, 0.5, 1.0, 1.5, 2.0]
    train_scores, val_scores = validation_curve(
        AdaBoostClassifier(n_estimators=best_n_estimators, random_state=42),
        X_train, y_train,
        param_name='learning_rate',
        param_range=learning_rate_range,
        cv=skf,
        scoring='accuracy'
    )

    best_learning_rate = learning_rate_range[np.argmax(val_scores.mean(axis=1))]
    print(f"Best learning_rate: {best_learning_rate}")

    # Tạo model tối ưu
    best_model = AdaBoostClassifier(
        n_estimators=best_n_estimators,
        learning_rate=best_learning_rate,
        random_state=42
    )

    # Cross-validation score
    cv_scores = cross_val_score(best_model, X_train, y_train, cv=skf, scoring='accuracy')
    print(f"CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

    return best_model

In [ ]:
# ===============================
# 5. BAYESIAN OPTIMIZATION (sử dụng Optuna)
# ===============================
def optimize_with_bayesian(X_train, y_train):
    """Bayesian Optimization với Optuna (nếu có)"""
    try:
        import optuna

        print("\n" + "=" * 60)
        print("OPTIMIZATION VỚI BAYESIAN (OPTUNA)")
        print("=" * 60)

        def objective(trial):
            # Định nghĩa hyperparameters để tối ưu
            n_estimators = trial.suggest_int('n_estimators', 50, 500)
            learning_rate = trial.suggest_float('learning_rate', 0.01, 2.0)
            max_depth = trial.suggest_int('max_depth', 1, 5)
            min_samples_split = trial.suggest_int('min_samples_split', 2, 20)

            # Tạo model
            base_estimator = DecisionTreeClassifier(
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                random_state=42
            )

            model = AdaBoostClassifier(
                estimator=base_estimator,
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                random_state=42
            )

            # Cross-validation
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy')
            return scores.mean()

        # Tạo study và optimize
        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=100)

        print(f"Best parameters: {study.best_params}")
        print(f"Best CV score: {study.best_value:.4f}")

        # Tạo best model
        best_params = study.best_params
        base_estimator = DecisionTreeClassifier(
            max_depth=best_params['max_depth'],
            min_samples_split=best_params['min_samples_split'],
            random_state=42
        )

        best_model = AdaBoostClassifier(
            estimator=base_estimator,
            n_estimators=best_params['n_estimators'],
            learning_rate=best_params['learning_rate'],
            random_state=42
        )

        return best_model

    except ImportError:
        print("\n" + "=" * 60)
        print("OPTUNA KHÔNG ĐƯỢC CÀI ĐẶT")
        print("Cài đặt bằng: pip install optuna")
        print("=" * 60)
        return None

In [ ]:
# ===============================
# 6. LEARNING CURVE ANALYSIS
# ===============================
def plot_learning_curves(models, X_train, y_train, model_names):
    """Vẽ learning curves cho các models"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()

    for idx, (model, name) in enumerate(zip(models, model_names)):
        if model is None:
            continue

        train_sizes, train_scores, val_scores = learning_curve(
            model, X_train, y_train,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
            train_sizes=np.linspace(0.1, 1.0, 10),
            scoring='accuracy'
        )

        train_mean = train_scores.mean(axis=1)
        train_std = train_scores.std(axis=1)
        val_mean = val_scores.mean(axis=1)
        val_std = val_scores.std(axis=1)

        axes[idx].plot(train_sizes, train_mean, 'o-', label='Training Score')
        axes[idx].fill_between(train_sizes, train_mean - train_std,
                              train_mean + train_std, alpha=0.1)

        axes[idx].plot(train_sizes, val_mean, 'o-', label='Validation Score')
        axes[idx].fill_between(train_sizes, val_mean - val_std,
                              val_mean + val_std, alpha=0.1)

        axes[idx].set_title(f'Learning Curve - {name}')
        axes[idx].set_xlabel('Training Set Size')
        axes[idx].set_ylabel('Accuracy Score')
        axes[idx].legend()
        axes[idx].grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# ===============================
# 7. EVALUATE MODELS
# ===============================
def evaluate_models(models, model_names, X_train, X_test, y_train, y_test):
    """Đánh giá các models"""
    print("\n" + "=" * 80)
    print("ĐÁNH GIÁ CÁC MODELS")
    print("=" * 80)

    results = []

    for model, name in zip(models, model_names):
        if model is None:
            continue

        # Fit model
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)

        # Accuracy
        accuracy = accuracy_score(y_test, y_pred)

        # Cross-validation score
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy')

        results.append({
            'Model': name,
            'Test Accuracy': accuracy,
            'CV Mean': cv_scores.mean(),
            'CV Std': cv_scores.std()
        })

        print(f"\n{name}:")
        print(f"Test Accuracy: {accuracy:.4f}")
        print(f"CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        print(f"Classification Report:")
        print(classification_report(y_test, y_pred))

    # Tạo DataFrame kết quả
    results_df = pd.DataFrame(results)
    print(f"\n{'='*50}")
    print("TỔNG HỢP KẾT QUẢ:")
    print(results_df.to_string(index=False))

    return results_df

In [ ]:
# ===============================
# 8. MAIN FUNCTION
# ===============================
def main():
    """Hàm chính"""
    print("ADABOOST OPTIMIZATION EXAMPLES")
    print("=" * 80)

    # Load data
    X_train, X_test, y_train, y_test = load_and_prepare_data()
    print(f"Dataset shape: {X_train.shape}")
    print(f"Number of classes: {len(np.unique(y_train))}")

    # Optimize với các phương pháp khác nhau
    models = []
    model_names = []

    # 1. StratifiedKFold Grid Search
    model1 = optimize_with_stratified_kfold(X_train, y_train)
    models.append(model1)
    model_names.append("StratifiedKFold GridSearch")

    # 2. Randomized Search
    model2 = optimize_with_randomized_search(X_train, y_train)
    models.append(model2)
    model_names.append("RandomizedSearch")

    # 3. Manual Optimization
    model3 = optimize_manually(X_train, y_train)
    models.append(model3)
    model_names.append("Manual Optimization")

    # 4. Bayesian Optimization
    model4 = optimize_with_bayesian(X_train, y_train)
    models.append(model4)
    model_names.append("Bayesian Optimization")

    # Evaluate all models
    results_df = evaluate_models(models, model_names, X_train, X_test, y_train, y_test)

    # Plot learning curves
    plot_learning_curves(models, X_train, y_train, model_names)

    return models, results_df